In [21]:
from root import ROOT_PATH
from pathlib import Path
import os

os.chdir(ROOT_PATH)
import pandas

from src.consts import ANNOTATED_BASE_PATH

result_path = ANNOTATED_BASE_PATH / "4axis"

p = Path("/home/rsoleyma/projects/LabelStudioHelper/data/annotations_tables/2")
csvs = sorted(p.glob("*.json")).pop()

In [22]:
df_orig = pandas.read_json(csvs, orient="records")

nicknames = {
    "ramin": "🤨",
    "roosmouthaan": "🤓",
    "lalala": "🤔",
    "priscilag.costa": "😤",
    "giulia.benati": "🤐",
    "xueyuan.liang": "😎",
    "fulvia.calcagni": "😏",
    "ariverca28": "😆"
}


def set_nicknames(lst):
    return [nicknames.get(item, item) for item in lst]


def make_anon(col, icon: str = "🎯"):
    if col == "":
        return col
    return "".join([icon for _ in col])


def merge(row):
    return "❓" * len(row["relevant-Uncertain"]) + "🎯" * len(row["relevant-Relevant"]) + "📭" * len(
        row["relevant-Not relevant"])  # "relevant-Uncertain"]


df = df_orig.copy()
df["post_text"] = df_orig["post_text"].apply(lambda x: x.replace("\n", ""))

# for col in df.columns:
#     df[col] = df[col].apply(lambda x: "" if pd.isna(x) else x)
# 
# # NICKNAMES
for col in df.columns[2:-1]:
    df[col] = df[col].apply(set_nicknames)

relevance_cols = ['relevant-Relevant',
                  'relevant-Not relevant',
                  'relevant-Uncertain']

landscape_cols = ['landscape-artificial surfaces',
                  'landscape-agricultural',
                  'landscape-Uncertain',
                  'landscape-forest and seminatural areas',
                  'landscape-wetlands',
                  'landscape-water bodies',
                  'landscape-not identifiable',
                  'landscape-ambigious']
dicho_cols = [
    'non_human-non-human',
    'non_human-human',
    'material-material',
    'material-ideel',
    'life-life',
    'life-mineral',
    'ideal_state-ideal_state',
    'ideal_state-alteration']

main_cols = relevance_cols + landscape_cols + dicho_cols

for col in relevance_cols:
    df[col + "_anon"] = df[col].apply(make_anon)


def nickname_notes(note):
    return {
        nicknames[k]: v for k, v in note.items()
    }


def anon_notes(note):
    return {
        f"‼️{idx}": v for idx, v in enumerate(note.values())
    }


df["notes"] = df["notes"].apply(nickname_notes)
df["notes_anon"] = df["notes"].apply(anon_notes)
df["merge"] = df.apply(merge, axis=1)

In [23]:
df.head()

,task,post_text,relevant-Relevant,relevant-Not relevant,relevant-Uncertain,relevant-000,landscape-artificial surfaces,landscape-agricultural,landscape-Uncertain,landscape-forest and seminatural areas,...,life-000,ideal_state-ideal_state,ideal_state-alteration,ideal_state-000,notes,relevant-Relevant_anon,relevant-Not relevant_anon,relevant-Uncertain_anon,notes_anon,merge
0,1056,7 people followed me and 3 people unfollowed m...,[],"[😆, 😏, 🤐, 😤, 🤨, 🤓, 😎]",[],[],[],[],[],[],...,"[😆, 😏, 🤐, 😤, 🤨, 🤓, 😎]",[],[],"[😆, 😏, 🤐, 😤, 🤨, 🤓, 😎]",{},,🎯🎯🎯🎯🎯🎯🎯,,{},📭📭📭📭📭📭📭
1,1057,Araujo's passes have been top notch today,[],"[😆, 🤐, 😤, 🤨, 🤓, 😎]",[],[😏],[],[],[],[],...,"[😆, 😏, 🤐, 😤, 🤨, 🤓, 😎]",[],[],"[😆, 😏, 🤐, 😤, 🤨, 🤓, 😎]",{},,🎯🎯🎯🎯🎯🎯,,{},📭📭📭📭📭📭
2,1058,What could be better than 3 rings in 1? #Gabr...,[],"[😆, 🤐, 😤, 🤨, 🤓, 😎]",[],[😏],[],[],[],[],...,"[😆, 😏, 🤐, 😤, 🤨, 🤓, 😎]",[],[],"[😆, 😏, 🤐, 😤, 🤨, 🤓, 😎]",{},,🎯🎯🎯🎯🎯🎯,,{},📭📭📭📭📭📭
3,1059,Aye everyone thanks for 400 follows… IN LESS T...,[],"[😆, 🤐, 😤, 🤨, 🤓, 😎]",[],[😏],[],[],[],[],...,"[😆, 😏, 🤐, 😤, 🤨, 🤓, 😎]",[],[],"[😆, 😏, 🤐, 😤, 🤨, 🤓, 😎]",{},,🎯🎯🎯🎯🎯🎯,,{},📭📭📭📭📭📭
4,1060,three solo wins in two days 😳,[],"[😆, 🤐, 😤, 🤨, 🤓, 😎]",[],[😏],[],[],[],[],...,"[😆, 😏, 🤐, 😤, 🤨, 🤓, 😎]",[],[],"[😆, 😏, 🤐, 😤, 🤨, 🤓, 😎]",{},,🎯🎯🎯🎯🎯🎯,,{},📭📭📭📭📭📭


In [24]:
rel_col, rel_colN, notes_col = ['relevant-Relevant', 'relevant-Uncertain', "notes"]

mask = [bool(set(a) | set(b) | set(c)) for a, b, c in zip(df[rel_col], df[rel_colN], df[notes_col])]

res1 = df[mask][["post_text", "merge", "notes_anon"]]


def create_anon_html():
    """
    only merge and notes
    """
    res1.to_html( result_path / "anon.html")


def create_normal_html():
    df_orig[mask][["post_text", "notes"] + main_cols].to_html( result_path / "normal.html")
    
create_anon_html()
create_normal_html()
    

In [16]:
res1

,post_text,merge,notes_anon
19,Just posted a photo @ New Zealand https://t.co...,❓📭📭📭📭📭,{}
25,Me and the boys pulling up to pinch mfers not ...,❓📭📭📭📭📭📭,{}
30,About Saturday. #grainy #photodiary https://t....,❓📭📭📭📭📭📭,{}
31,Is Slut Pop by Kim Petras going to be summer a...,🎯📭📭📭📭📭,{'‼️0': 'cultural product can create positive ...
32,#savesoil@sadhgurujv@cpconciousplanet Kiss The...,🎯🎯🎯🎯🎯🎯,{'‼️0': 'the video of this tweet is also about...
...,...,...,...
950,Images allegedly show construction at North Ko...,🎯📭📭📭,{}
963,On 16/02/22 at 04:45 the river level was 0.15m...,🎯🎯🎯📭📭,{}
968,$1 (🌏 haters !!!!!!!!!!!!!!!!!!!!!!!!. https:/...,❓📭📭📭📭📭,{}
975,Dog eventually is going to get the pup mask,❓❓📭📭📭📭,{'‼️0': 'Not sure whether it is a metaphore'}


In [17]:
rel_col, rel_colN, notes_col = ['relevant-Relevant', 'relevant-Uncertain', "notes"]

res2 = df[mask][["post_text", "merge", "notes"] + main_cols]

In [28]:

### MAKE THIS NICER. APPLY TO NORMAL AS WELL
styled_df = res2.style.set_properties(**{
    'background-color': '#cfcbcb',
    'color': 'black',
    'border': '1px solid black',
    'class': 'custom-class'
}, subset=relevance_cols).set_properties(**{
    'background-color': '#cfeb3b',
    'color': 'black',
    'border': '1px solid black',
    'class': 'custom-class'
}, subset=["landscape-artificial surfaces", 'landscape-agricultural',
           'landscape-Uncertain',
           'landscape-forest and seminatural areas',
           'landscape-wetlands',
           'landscape-water bodies',
           'landscape-not identifiable',
           'landscape-ambigious']).set_properties(**{
    'background-color': '#afebbb',
    'color': 'black',
    'border': '1px solid black',
    'class': 'custom-class'
}, subset=['non_human-non-human',
           'non_human-human']).set_properties(**{
    'background-color': '#ff2b5b',
    'color': 'black',
    'border': '1px solid black',
    'class': 'custom-class'
}, subset=['material-material',
           'material-ideel']).set_properties(**{
    'background-color': '#0febeb',
    'color': 'black',
    'border': '1px solid black',
    'class': 'custom-class'
}, subset=['life-life',
           'life-mineral']).set_properties(**{
    'background-color': '#afeb7b',
    'color': 'black',
    'border': '1px solid black',
    'class': 'custom-class'
}, subset=[
    'ideal_state-ideal_state',
    'ideal_state-alteration'])

# Combine with CSS
html = styled_df.to_html()
css = '''
<style>
.custom-class {
    font-weight: bold;
    padding: 10px;
}
</style>
'''
complete_html = css + html
(result_path / "full-icons.html").write_text(complete_html)
print((ANNOTATED_BASE_PATH / "full-icons.html").absolute())

/home/rsoleyma/projects/twitter-stream-unpacker/data/annotated/full-icons.html


In [29]:
import markdown

dichotomies = [
    "relevant-Relevant"
]


def get_task_html(row, emoji: bool = False):
    def mod_userlist(lst: list[str]) -> str:
        if not lst:
            return "✖️"
        if emoji:
            return "".join([nicknames[c] for c in lst])
        else:
            return ", ".join(lst)

    s = []
    s.append(f"## {row['post_text']}")
    for u in ["relevant-Relevant", "relevant-Not relevant", "relevant-Uncertain"]:
        if row[u]:
            s.append(f"__{u}:__ {mod_userlist(row[u])}")
    s.append("")
    s.append("### Landscape")
    for u in ['landscape-artificial surfaces',
              'landscape-agricultural',
              'landscape-Uncertain',
              'landscape-forest and seminatural areas',
              'landscape-wetlands',
              'landscape-water bodies',
              'landscape-not identifiable',
              'landscape-ambigious']:
        if row[u]:
            s.append(f"__{u}:__ {mod_userlist(row[u])}")

    s.append("### 4 Axis")
    ## here get the proper pandas fraction and create a normal table.
    for u1, u2 in [['non_human-non-human',
                    'non_human-human'],
                   ['material-material',
                    'material-ideel'],
                   ['life-life',
                    'life-mineral'],
                   ['ideal_state-ideal_state',
                    'ideal_state-alteration']]:
        #pd.DataFrame([u1,)
        si = u1.index("-")
        name = u1[:si]
        u1n = u1[si + 1:]
        u2n = u2[si + 1:]
        alert = 1 if u1n and u2n else ""

        s.append(f"__{name} {alert}::: {u1n}:__ {mod_userlist(row[u1])} __{u2n}:__ {mod_userlist(row[u2])}")
    if row["notes"]:
        s.append("### Notes")
        for coder, note in row["notes"].items():
            s.append(f"__{coder}:__ {note}")
    return "\n\n".join(s)


with open(result_path / "easy-emo.html", "w", encoding="utf-8") as f:
    all_lines = []
    for row in df_orig[mask].to_dict('records'):
        all_lines.append(get_task_html(row, True))
    res = "\n\n".join(all_lines)
    html = markdown.markdown(res)
    f.write(html)

with open(result_path / "easy.html", "w", encoding="utf-8") as f:
    all_lines = []
    for row in df_orig[mask].to_dict('records'):
        all_lines.append(get_task_html(row, False))
    res = "\n\n".join(all_lines)
    html = markdown.markdown(res)
    f.write(html)